# TalkTalk Retention — Data Quality Audit

Reproduces the `/analysis` page on full data. Drop your four files into `./data/`:
- `customer_info.parquet` (or `.csv`)
- `usage.parquet` (or `.csv`)
- `calls.csv`
- `cease.csv`

Then run all cells. Output: charts + `out/audit_report.json` summary.

In [ ]:
# 0. Setup
import pandas as pd, numpy as np, json, os
from pathlib import Path
import matplotlib.pyplot as plt
try:
    import missingno as msno
except ImportError:
    msno = None
DATA = Path('./data')
OUT  = Path('./out'); OUT.mkdir(exist_ok=True)
def load(name):
    for ext in ('.parquet', '.csv'):
        p = DATA / f'{name}{ext}'
        if p.exists():
            return pd.read_parquet(p) if ext == '.parquet' else pd.read_csv(p)
    raise FileNotFoundError(name)
ci, usage, calls, cease = load('customer_info'), load('usage'), load('calls'), load('cease')
print({k: len(v) for k,v in dict(customer_info=ci, usage=usage, calls=calls, cease=cease).items()})

## 1. Coverage & shape

In [ ]:
for name, df, datecol in [('customer_info',ci,'datevalue'),('usage',usage,'calendar_date'),
                                ('calls',calls,'event_date'),('cease',cease,'cease_placed_date')]:
    df[datecol] = pd.to_datetime(df[datecol], errors='coerce')
    print(f'{name:14s}  {df[datecol].min().date()} -> {df[datecol].max().date()}  rows={len(df):>10,}  customers={df["unique_customer_identifier"].nunique():>8,}')

## 2. Cleanliness

In [ ]:
null_rates = {n: (df.isna().mean()*100).round(2).to_dict()
               for n, df in [('customer_info',ci),('usage',usage),('calls',calls),('cease',cease)]}
print(json.dumps(null_rates, indent=2, default=str))
if msno: msno.matrix(ci.sample(min(2000,len(ci))))

In [ ]:
zeros = (ci['line_speed']==0).mean()*100
print(f'line_speed == 0 on {zeros:.2f}% of rows  (treat as missing!)')
for col in ('usage_download_mbs','usage_upload_mbs'):
    bad = pd.to_numeric(usage[col], errors='coerce').isna().sum()
    print(f'{col}: {bad} rows fail numeric cast')

## 3. Cease reason quality

In [ ]:
r = cease['reason_description_insight'].value_counts()
print(r); r.plot.bar(title='Cease reason distribution'); plt.show()
vague = (cease['reason_description_insight']=='VagueReason').mean()*100
print(f'>>> {vague:.0f}% of ceases are VagueReason — root-cause signal is poor.')

## 4. Call-type mix

In [ ]:
ct = calls['call_type'].fillna('(null)').value_counts(); print(ct)
ct.plot.bar(title='Call types'); plt.show()

## 5. Rolling-window features the live model should use

In [ ]:
usage = usage.sort_values(['unique_customer_identifier','calendar_date'])
usage['dl'] = pd.to_numeric(usage['usage_download_mbs'], errors='coerce')
g = usage.groupby('unique_customer_identifier')['dl']
usage['dl_30d'] = g.transform(lambda s: s.rolling(30, min_periods=5).mean())
usage['dl_90d'] = g.transform(lambda s: s.rolling(90, min_periods=20).mean())
usage['dl_drop'] = (usage['dl_30d']/usage['dl_90d'] - 1).clip(-1,1)
print(usage[['dl_30d','dl_90d','dl_drop']].describe())

## 6. Label-leakage check

In [ ]:
m = cease[['unique_customer_identifier','cease_placed_date']].merge(
        ci[['unique_customer_identifier','datevalue']], on='unique_customer_identifier')
leak = (m['datevalue'] > pd.to_datetime(m['cease_placed_date'])).sum()
print(f'customer_info rows dated after cease: {leak} (should be 0 for training set)')

## 7. PSI drift between two date windows

In [ ]:
def psi(a, b, bins=10):
    breaks = np.unique(np.quantile(a, np.linspace(0,1,bins+1)))
    a_pct = np.histogram(a, breaks)[0]/len(a) + 1e-6
    b_pct = np.histogram(b, breaks)[0]/len(b) + 1e-6
    return float(((a_pct-b_pct)*np.log(a_pct/b_pct)).sum())
mid = ci['datevalue'].median()
old = ci.loc[ci['datevalue'] <  mid, 'tenure_days'].dropna()
new = ci.loc[ci['datevalue'] >= mid, 'tenure_days'].dropna()
print('PSI(tenure_days) old vs new =', round(psi(old,new),4), '   (>0.2 = significant drift)')

## 8. Save audit report

In [ ]:
report = dict(rows={k:len(v) for k,v in dict(ci=ci,usage=usage,calls=calls,cease=cease).items()},
              null_rates=null_rates, line_speed_zero_pct=float(zeros),
              cease_vague_pct=float(vague), psi_tenure=psi(old,new))
(OUT/'audit_report.json').write_text(json.dumps(report, indent=2, default=str))
print('wrote out/audit_report.json')